# LightOnOCR-3 0.8B — document OCR on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/LightOnOCR-3-0.8B-document-ocr.ipynb)

[`lightonai/LightOnOCR-3-0.8B`](https://huggingface.co/lightonai/LightOnOCR-3-0.8B) is LightOn's smallest OCR model, Apache 2.0, built on the Qwen3.5 vision-language architecture (about 0.8B parameters, 1.7 GB of weights). An empty prompt transcribes the page to markdown. The prompt `grounding` adds a label and a bounding box, in page coordinates normalized to 0–1000, for every block: text, tables, handwriting, and charts (charts come back as an HTML table of the plotted values). One model replaces a detection-plus-recognition pipeline.

**Licence:** Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has 16 GB of VRAM, no native bfloat16, and no FlashAttention 2. This notebook loads the checkpoint in **float16** with **SDPA** attention (eager if SDPA refuses the hybrid stack). Weights are about 1.7 GB; a page whose longest edge is 2048 px peaks at roughly 5–8 GB. That estimate is from the weight file plus a 2048 px page, not a measured Colab run.

**Not in this notebook:** a LoRA fine-tune. The model card has no training recipe, `use_cache` is saved as false, and the text stack mixes full attention with linear attention. A short LoRA step on a document-sized image does not leave a comfortable margin on 16 GB, so there is no training cell.

**Links:** [model card](https://huggingface.co/lightonai/LightOnOCR-3-0.8B) · [blog](https://huggingface.co/blog/lightonai/lightonocr-3) · [paper](https://arxiv.org/abs/2601.14251) · [demo](https://huggingface.co/spaces/lightonai/LightOnOCR-3-Demo)


In [ ]:
%pip install -q "transformers==5.20.1" "pypdfium2==5.14.0" pillow

# The model card requires transformers>=5.5.4 (the checkpoint was saved with 5.5.4)
# and loads as Qwen3_5ForConditionalGeneration. 5.20.1 is a pinned release above that floor.
# pypdfium2 renders the PDF page. pillow is the image library the card lists; Colab already has it.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.5.4"), (
    f"transformers {transformers.__version__} is loaded; LightOnOCR-3 needs 5.5.4+. "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2. Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


## 1. Load LightOnOCR-3 0.8B

The card loads `Qwen3_5ForConditionalGeneration` and `AutoProcessor`. Thinking stays off: the checkpoint is trained and evaluated that way, and the chat template defaults to it. The first run downloads about 1.7 GB.


In [ ]:
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

MODEL_ID = "lightonai/LightOnOCR-3-0.8B"


def load_ocr(attn):
    return Qwen3_5ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        attn_implementation=attn,
        device_map="auto",
    ).eval()


# SDPA is the T4 path. Fall back to eager if this transformers build rejects SDPA
# on Qwen3.5's linear-attention layers.
try:
    model = load_ocr("sdpa")
    ATTN = "sdpa"
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    model = load_ocr("eager")
    ATTN = "eager"

processor = AutoProcessor.from_pretrained(MODEL_ID)
print("attention:", ATTN, "| dtype:", model.dtype)
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## 2. Receipt, handwriting, and a results table

Samples:

* the SROIE receipt from the model card
* a handwritten resignation letter ([CIPHR Connect](https://commons.wikimedia.org/wiki/File:Handwritten_Letter_of_Resignation.jpg), CC BY 2.0)
* page 8 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (arXiv:1706.03762), which holds Table 2

PDFs are rendered at 400 DPI and limited to a 2048 px longest edge, which is the card's preprocessing recipe. The grounding prompt is the only instruction the card supports besides an empty transcription prompt.


In [ ]:
import io
import re

import requests
from IPython.display import display
from PIL import Image, ImageDraw

RECEIPT_URL = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"
LETTER_URL = "https://upload.wikimedia.org/wikipedia/commons/e/eb/Handwritten_Letter_of_Resignation.jpg"
PAPER_URL = "https://arxiv.org/pdf/1706.03762"
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}
BLOCK = re.compile(r"!\[([^\]]+)\]\((\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\)")


def fetch_image(url):
    response = requests.get(url, headers=UA, timeout=120)
    response.raise_for_status()
    return Image.open(io.BytesIO(response.content)).convert("RGB")


def render_pdf_page(url, page_index, longest=2048):
    import pypdfium2 as pdfium

    response = requests.get(url, headers=UA, timeout=120)
    response.raise_for_status()
    pdf = pdfium.PdfDocument(response.content)
    image = pdf[page_index].render(scale=400 / 72).to_pil().convert("RGB")
    image.thumbnail((longest, longest))
    return image


def limit_edge(image, longest):
    image = image.copy()
    image.thumbnail((longest, longest))
    return image


receipt = limit_edge(fetch_image(RECEIPT_URL), 2048)
letter = limit_edge(fetch_image(LETTER_URL), 2048)
table_page = render_pdf_page(PAPER_URL, page_index=7)  # printed page 8, Table 2
print("receipt", receipt.size, "| letter", letter.size, "| table page", table_page.size)


def to_model(inputs):
    inputs = inputs.to(model.device)
    for key, value in list(inputs.items()):
        if hasattr(value, "is_floating_point") and value.is_floating_point():
            inputs[key] = value.to(dtype=model.dtype)
    return inputs


def transcribe(image, prompt, max_new_tokens=640):
    # URL strings follow the model card. Local pages pass the PIL image.
    if isinstance(image, str):
        content = [{"type": "image", "url": image}]
    else:
        content = [{"type": "image", "image": image}]
    if prompt:
        content.append({"type": "text", "text": prompt})
    conversation = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    inputs = to_model(inputs)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
        )
    generated = output_ids[0, inputs["input_ids"].shape[1] :]
    text = processor.decode(generated, skip_special_tokens=True)
    if torch.cuda.is_available():
        print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    return text


def draw_grounding(image, text):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    width, height = canvas.size
    hits = list(BLOCK.finditer(text))
    for match in hits:
        label, x1, y1, x2, y2 = match.groups()
        box = (
            int(x1) / 1000 * width,
            int(y1) / 1000 * height,
            int(x2) / 1000 * width,
            int(y2) / 1000 * height,
        )
        draw.rectangle(box, outline=(220, 40, 40), width=3)
        draw.text((box[0] + 2, box[1] + 2), label, fill=(220, 40, 40))
    return canvas, len(hits)


### Receipt — transcription, then grounding

The first call is the model card's example: image URL, prompt `grounding`, temperature 0.2. The picture below is the same file, with the returned boxes drawn on it. Coordinates are 0–1000, not pixels.


In [ ]:
receipt_text = transcribe(RECEIPT_URL, "grounding")
print(receipt_text)
boxed, n_boxes = draw_grounding(receipt, receipt_text)
print(f"{n_boxes} grounding boxes")
display(boxed)


### Handwriting


In [ ]:
print("Source: CIPHR Connect, CC BY 2.0, via Wikimedia Commons.")
display(letter)
letter_text = transcribe(letter, "grounding", max_new_tokens=800)
print(letter_text)
boxed, n_boxes = draw_grounding(letter, letter_text)
print(f"{n_boxes} grounding boxes")
display(boxed)


### Table — page 8 of Attention Is All You Need


In [ ]:
display(table_page)
table_text = transcribe(table_page, "grounding", max_new_tokens=900)
print(table_text)
boxed, n_boxes = draw_grounding(table_page, table_text)
print(f"{n_boxes} grounding boxes")
display(boxed)
